# Actividad 2026 · Unidad 1

**Atribución:** esta actividad adapta material del [Deep Reinforcement Learning Course de Hugging Face](https://huggingface.co/learn/deep-rl-course). El contenido original pertenece a sus autores y colaboradores.

**Traducción:** texto en español elaborado con asistencia de IA a partir del material original. Puede contener errores; ante una duda, consultá el notebook fuente.

[Abrir esta unidad en Colab](https://colab.research.google.com/drive/176nZrSvdgZe84tedHzRTpMX80LBE3S5O)

Adaptación del [notebook oficial](https://github.com/huggingface/deep-rl-class/blob/06d62b08e046ef976022ad3cc8fc2a3c15d6af10/notebooks/unit1/unit1.ipynb). Los TODO son ejercicios; las celdas de publicación en el Hub son optativas y requieren cuenta.
Entrenamiento PPO sobre LunarLander-v3. La ejecución completa usa un millón de pasos; el Hub se omite.


# Unidad 1: entrená tu primer agente de aprendizaje por refuerzos profundo 🤖

![Portada](https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit1/thumbnail.jpg)

En este notebook vas a entrenar un agente para **aterrizar en la Luna 🌕** con [Stable-Baselines3](https://stable-baselines3.readthedocs.io/en/master/). También vas a experimentar con distintas configuraciones.

⬇️ Este es un ejemplo del resultado que buscamos. ⬇️


**Ejercicio: completar antes de ejecutar.** La solución aparece más adelante cuando está incluida en el original.

```python
%%html
<video controls autoplay><source src="https://huggingface.co/sb3/ppo-LunarLander-v3/resolve/main/replay.mp4" type="video/mp4"></video>
```


### Entorno 🎮

- [LunarLander-v3](https://gymnasium.farama.org/environments/box2d/lunar_lander/)

### Biblioteca 📚

- [Stable-Baselines3](https://stable-baselines3.readthedocs.io/en/master/)


Si encontrás un problema en el material original, podés [abrir un issue en su repositorio](https://github.com/huggingface/deep-rl-class/issues).

## Objetivos 🏆

Al terminar, vas a poder:

- Usar **Gymnasium** para crear y explorar un entorno.
- Entrenar un agente con **Stable-Baselines3**.
- Guardar y evaluar el agente. La publicación en Hugging Face Hub queda fuera de esta adaptación.


## Material del curso de aprendizaje por refuerzos profundo de Hugging Face

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/deep-rl-course-illustration.jpg" alt="Ilustración del curso de aprendizaje por refuerzos profundo"/>


En el [curso gratuito de Hugging Face](https://huggingface.co/learn/deep-rl-course) se estudia aprendizaje por refuerzos profundo en teoría y práctica, se usan bibliotecas como Stable-Baselines3 y RL Baselines3 Zoo, y se entrenan agentes en distintos entornos.

Podés consultar el [programa original](https://simoninithomas.github.io/deep-rl-course) y participar en su [comunidad de Discord](https://discord.gg/ydHrjt3WP5). El curso original también ofrece actividades y un certificado de finalización.


## Requisitos previos 🏗️

Antes de empezar, leé la [unidad 0](https://huggingface.co/deep-rl-course/unit0/introduction), que presenta el curso, y la [unidad 1](https://huggingface.co/deep-rl-course/unit1/introduction), que introduce los fundamentos del aprendizaje por refuerzos.


## Repaso breve de aprendizaje por refuerzos profundo 📚

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit1/RL_process_game.jpg" alt="Proceso de aprendizaje por refuerzos" width="100%">


Recordemos las ideas principales de la unidad 1:

- El aprendizaje por refuerzos estudia cómo aprender de las acciones. Un agente interactúa con el entorno mediante prueba y error y recibe recompensas como señal.
- El objetivo es **maximizar la recompensa acumulada esperada**, también llamada retorno esperado.
- La interacción produce una secuencia de **estado, acción, recompensa y estado siguiente**.
- Al calcular el retorno, descontamos las recompensas futuras: las más lejanas suelen ser menos previsibles.
- Para resolver el problema buscamos una **política óptima**, que indique qué acción tomar en cada estado para maximizar el retorno esperado.

Hay dos enfoques habituales:

- **Métodos basados en políticas:** entrenan la política directamente.
- **Métodos basados en valores:** estiman el retorno esperado y usan esa estimación para definir la política.

En aprendizaje profundo por refuerzos se emplean redes neuronales para aproximar la política o la función de valor.


In [ ]:
!apt install swig cmake

In [ ]:
%pip install -q 'gymnasium[box2d]' stable-baselines3


In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor


## Gymnasium: qué es y cómo funciona 🤖

[Gymnasium](https://farama.org/) es una biblioteca mantenida por la Farama Foundation que continúa el desarrollo de Gym. La vamos a usar a menudo en aprendizaje por refuerzos.

Ofrece una interfaz para **crear entornos** y una **colección de entornos** de control, Atari y Box2D, entre otros.

Antes de ver el código, recordemos el ciclo de interacción:

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit1/RL_process_game.jpg" alt="Proceso de aprendizaje por refuerzos" width="100%">


En cada paso, el agente recibe un **estado** del entorno, elige una **acción**, y el entorno devuelve un **estado nuevo** y una **recompensa**.

Con Gymnasium:

1. Creamos el entorno con `gymnasium.make()`.
2. Lo reiniciamos con `observation, info = env.reset()`.
3. Elegimos una acción; en el ejemplo siguiente será aleatoria.
4. Ejecutamos `env.step(action)`, que devuelve:
   - `observation`: estado siguiente;
   - `reward`: recompensa;
   - `terminated`: indica que se alcanzó un estado terminal;
   - `truncated`: indica un corte externo, por ejemplo un límite de tiempo;
   - `info`: datos adicionales del entorno.

Al terminar o truncarse el episodio, reiniciamos el entorno. La [documentación de `step`](https://gymnasium.farama.org/api/env/#gymnasium.Env.step) amplía estos conceptos.

**Veamos el ejemplo y leamos su código.**


In [ ]:
import gymnasium as gym

# Primero creamos el entorno LunarLander-v3
env = gym.make("LunarLander-v3")

# Después reiniciamos el entorno
observation, info = env.reset()

for _ in range(20):
  # Elegimos una acción al azar
  action = env.action_space.sample()
  print("Acción elegida:", action)

  # Ejecutamos la acción y obtenemos
  # observación, recompensa, terminated, truncated e info
  observation, reward, terminated, truncated, info = env.step(action)

  # Si el episodio termina (aterrizaje o choque) o se trunca (límite de tiempo)
  if terminated or truncated:
      # Reiniciamos el entorno
      print("Entorno reiniciado")
      observation, info = env.reset()

env.close()

## Creá el entorno LunarLander 🌛

En este tutorial entrenaremos al agente de [LunarLander-v3](https://gymnasium.farama.org/environments/box2d/lunar_lander/) para que **aterrice correctamente**. Debe ajustar su posición y sus velocidades horizontal, vertical y angular.

💡 Antes de usar un entorno, conviene consultar su [documentación](https://gymnasium.farama.org/environments/box2d/lunar_lander/).


Veamos cómo es el entorno:

In [ ]:
# Creamos el entorno con gym.make("nombre_del_entorno")
env = gym.make("LunarLander-v3")
env.reset()
print("_____ESPACIO DE OBSERVACIÓN_____ \n")
print("Forma del espacio de observación", env.observation_space.shape)
print("Observación de ejemplo", env.observation_space.sample()) # Obtener una observación aleatoria

`Forma del espacio de observación (8,)` indica que cada observación es un vector de ocho valores:

- posición horizontal y vertical;
- velocidad horizontal y vertical;
- ángulo y velocidad angular;
- contacto de la pata izquierda y de la pata derecha con el suelo.


In [ ]:
print("\n _____ESPACIO DE ACCIONES_____ \n")
print("Cantidad de acciones", env.action_space.n)
print("Acción de ejemplo", env.action_space.sample()) # Elegimos una acción al azar

El espacio de acciones es discreto y tiene cuatro opciones 🎮:

- Acción 0: no hacer nada.
- Acción 1: encender el motor lateral izquierdo.
- Acción 2: encender el motor principal.
- Acción 3: encender el motor lateral derecho.

La **recompensa total** del episodio es la suma de las recompensas de cada paso. La recompensa aumenta cuando el módulo se acerca a la plataforma y reduce su velocidad; disminuye si se inclina o usa combustible. Cada pata que toca el suelo aporta 10 puntos. Los motores laterales cuestan 0,03 puntos por cuadro y el principal, 0,3. Chocar añade −100 puntos y aterrizar de forma segura, +100.

Se considera resuelto un episodio que alcanza al menos **200 puntos**.


#### Entorno vectorizado

Creamos 16 entornos independientes y los agrupamos para obtener experiencias más variadas durante el entrenamiento.


In [ ]:
# Crear el entorno
env = make_vec_env('LunarLander-v3', n_envs=16)

## Creá el modelo 🤖

Ya conocemos el objetivo: controlar los motores para aterrizar en la plataforma. Usaremos [Stable-Baselines3 (SB3)](https://stable-baselines3.readthedocs.io/en/master/), una colección de implementaciones de algoritmos de aprendizaje por refuerzos en PyTorch.

💡 Al empezar con una biblioteca nueva, consultá su [documentación](https://stable-baselines3.readthedocs.io/en/master/) y sus ejemplos.


<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit1/sb3.png" alt="Stable Baselines3">

Usaremos **PPO** (*Proximal Policy Optimization*), un algoritmo que combina una estimación del valor con una política que asigna probabilidades a las acciones. Podés consultar la [documentación de PPO en SB3](https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html).


Para entrenar con Stable-Baselines3:

1. Creá el entorno.
2. Instanciá el modelo, por ejemplo `PPO("MlpPolicy", env)`.
3. Entrenalo con `model.learn(total_timesteps=...)`.

```python
# Crear el entorno
env = gym.make("LunarLander-v3")
# Crear y entrenar el agente
model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=int(2e5))
```


**Ejercicio: completar antes de ejecutar.** La solución aparece más adelante cuando está incluida en el original.

```python
# TODO: definir una arquitectura PPO con MlpPolicy
# Usamos MlpPolicy porque la entrada es un vector;
# si fueran imágenes, usaríamos CnnPolicy.
model =
```


#### Solución

In [ ]:
# @title Solución (Mostrar código) { display-mode: "form" }
# SOLUCIÓN
# Ajustamos algunos parámetros para acelerar el entrenamiento
model = PPO(
    policy = 'MlpPolicy',
    env = env,
    n_steps = 1024,
    batch_size = 64,
    n_epochs = 4,
    gamma = 0.999,
    gae_lambda = 0.98,
    ent_coef = 0.01,
    verbose=1)

## Entrená el agente PPO 🏃

La configuración usa un millón de pasos. En Colab, una ejecución larga puede beneficiarse de GPU; para comprobar el flujo podés entrenar menos pasos. El rendimiento de la política requiere una evaluación posterior.


**Extensión o ejercicio del notebook original; se omite en la ejecución 2026.**


#### Solución

In [ ]:
# @title Solución (Mostrar código) { display-mode: "form" }
# SOLUCIÓN
# Entrenar durante un millón de pasos
model.learn(total_timesteps=1000000)
# Guardar el modelo
model_name = "ppo-LunarLander-v3"
model.save(model_name)

## Evaluá el agente 📈

Una vez entrenado, medí su desempeño con `evaluate_policy` de Stable-Baselines3. Creá un **entorno de evaluación distinto** y envolvelo en [`Monitor`](https://stable-baselines3.readthedocs.io/en/master/common/monitor.html). Consultá el [ejemplo de evaluación](https://stable-baselines3.readthedocs.io/en/master/guide/examples.html#basic-usage-training-saving-loading).


**Ejercicio: completar antes de ejecutar.** La solución aparece más adelante cuando está incluida en el original.

```python
# TODO: evaluar el agente
# Crear un entorno nuevo para la evaluación
eval_env =

# Evaluar el modelo en 10 episodios con deterministic=True
mean_reward, std_reward =

# Mostrar los resultados


```


#### Solución

In [ ]:
# @title Solución (Mostrar código) { display-mode: "form" }
eval_env = Monitor(gym.make("LunarLander-v3", render_mode='rgb_array'))
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f"recompensa_media={mean_reward:.2f} +/- {std_reward}")

## Para discutir
La evaluación de PPO requiere entrenamiento suficiente. La prueba corta de 2048 pasos solo comprueba que el flujo funciona; el entrenamiento configurado arriba usa un millón de pasos.
